# Day 9 · Exercise 2: Design a Contact Schema

**What you'll build:** `extract_contact(text: str, model: str) -> dict` — a function that defines a `ContactCard` Pydantic schema with required fields, optional fields, and a nested `Address` model, then uses it to extract structured contact details from free text.

**Why it matters:** Schema design — choosing which fields are required vs optional, how to group them with nesting, and how to write precise `Field(description=...)` values — directly controls extraction accuracy; a well-designed schema is clearer instructions to the model than any prompt tweak.

## Your Implementation

In [ ]:
import json

import ollama
from pydantic import BaseModel, Field


def extract_contact(text: str, model: str) -> dict:
    """Extract structured contact information from free text using a Pydantic schema.

    Defines a ContactCard schema with:
      - Required top-level fields: name, email
      - Optional top-level fields: phone, website
      - Optional nested Address model: street, city, country
    Injects the JSON Schema into the system prompt via model_json_schema(),
    calls the Ollama model with format='json', validates with Pydantic, and
    returns the result as a dict.

    Args:
        text:  Free-form text that may contain contact details (e.g. a business
               card paragraph, an email signature, a plain sentence).
        model: Ollama model name to use, e.g. 'llama3.2'.

    Returns:
        A dict with keys matching the ContactCard schema.  Optional fields that
        are absent in the text will be None.  The 'address' key will be None if
        no address information is present.

    Example:
        result = extract_contact(
            "Jane Doe — jane@example.com | +1-555-0199",
            model="llama3.2",
        )
        # result == {
        #     'name': 'Jane Doe',
        #     'email': 'jane@example.com',
        #     'phone': '+1-555-0199',
        #     'website': None,
        #     'address': None,
        # }
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(extract_contact), 'extract_contact is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: extract_contact is defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a dict
    try:
        result = extract_contact(
            "Reach me at grace@navy.mil — I'm Grace Hopper.",
            model='llama3.2',
        )
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: extract_contact returns a dict')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: required fields present with correct values
    try:
        assert 'name' in result, "'name' key missing from result"
        assert 'email' in result, "'email' key missing from result"
        name_ok = 'hopper' in str(result.get('name', '')).lower() or 'grace' in str(result.get('name', '')).lower()
        email_ok = 'grace@navy.mil' in str(result.get('email', ''))
        assert name_ok, f"name not extracted correctly, got {result.get('name')!r}"
        assert email_ok, f"email not extracted correctly, got {result.get('email')!r}"
        print(f'{_PASS} Check 3/{total}: required fields name and email extracted correctly')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: optional fields present and absent fields are None (not missing)
    try:
        result2 = extract_contact(
            "Contact: Ada Lovelace, ada@lovelace.io, +44-20-7946-0101",
            model='llama3.2',
        )
        assert isinstance(result2, dict), f'expected dict, got {type(result2).__name__}'
        assert 'phone' in result2, "'phone' key missing — optional fields must be present (as None if absent)"
        assert 'website' in result2, "'website' key missing — optional fields must be present (as None if absent)"
        assert 'address' in result2, "'address' key missing — nested model key must always be present"
        phone_val = result2.get('phone')
        assert phone_val is not None, "phone should be extracted from the text, got None"
        print(f'{_PASS} Check 4/{total}: optional fields present; phone extracted, website/address None when absent')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Add a `tags: list[str]` field to `ContactCard` — foreshadowing Day 10's multi-value extraction.
Set it with `default_factory=list` and a description like:
`"Short labels describing the contact's role, e.g. ['engineer', 'open-source'] — empty list if none are implied."`

Then test it on this text and see what the model infers:

```python
result = extract_contact(
    "Dr. Sofia Esposito, CTO at Quantum Labs. sofia@quantum.io — open source contributor and AI researcher.",
    model="llama3.2",
)
print(result.get("tags"))  # e.g. ['CTO', 'AI researcher', 'open source']
```

This is the pattern for extracting collections of values — a skill you will use heavily in later days.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json

import ollama
from pydantic import BaseModel, Field


class Address(BaseModel):
    street: str | None = Field(
        default=None,
        description="Street address including number and street name, or null if not present",
    )
    city: str | None = Field(
        default=None,
        description="City or town name, or null if not present",
    )
    country: str | None = Field(
        default=None,
        description="Country name or two-letter ISO code, or null if not present",
    )


class ContactCard(BaseModel):
    name: str = Field(description="Full name of the person")
    email: str = Field(description="Primary email address")
    phone: str | None = Field(
        default=None,
        description="Phone number in any format, or null if not present in the text",
    )
    website: str | None = Field(
        default=None,
        description="Personal or company website URL, or null if not present",
    )
    address: Address | None = Field(
        default=None,
        description="Postal address details, or null if no address information is present",
    )


def extract_contact(text: str, model: str) -> dict:
    schema = ContactCard.model_json_schema()

    system_prompt = (
        "Extract contact information from the text.\n"
        "Return ONLY valid JSON matching this schema — no prose, no markdown:\n\n"
        + json.dumps(schema, indent=2)
        + "\n\nIf a field is not present in the text, set it to null."
    )

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        format="json",
    )

    raw = response["message"]["content"]
    contact = ContactCard.model_validate_json(raw)
    return contact.model_dump()
```

**Why this works:** `model_json_schema()` serialises every `Field(description=...)` into the JSON Schema that lands in the system prompt, turning your Python annotations into precise model instructions. Marking `phone`, `website`, and `address` as `str | None` / `Address | None` with `default=None` lets the model return `null` honestly when data is absent — preventing hallucination. Calling `model_dump()` at the end converts the validated Pydantic instance back to a plain dict, which is what the function signature promises to return.
</details>